# Topic: File Truncation - Truncate operations, null-byte pads, and buffer flushing (flush)
 
## 1. TRUNCATING FILES: truncate()
- **`f.truncate(size=None)`**:
  - Resizes the file to the specified `size` in bytes.
  - **Shrinking**: If the file is currently larger than `size`, all data past the size limit is discarded (deleted).
  - **Expanding**: If the file is currently smaller than `size`, it is padded to match the new size. The padded segment is filled with null bytes (`\x00`).
  - **Default size**: If `size` is omitted or `None`, it truncates the file to match the current file pointer position (from `tell()`).
- **Prerequisite**: The file must be opened in a writeable mode (e.g. `w`, `r+`, `a`).
 
## 2. FLUSHING BUFFERS: flush()
- **`f.flush()`**:
  - Forces writing any buffered data in Python's internal memory buffer wrapper to the underlying OS file descriptor buffer.
  - **Difference from close()**: `flush()` keeps the file object open and active, while ensuring data is written immediately (useful for logs or real-time file updates).
  - *OS Flush Note*: Calling `flush()` pushes data to the OS cache. To guarantee physical disk write, you must call `os.fsync(f.fileno())` after flushing.
 
---


In [ ]:
import os

truncate_file = "trunc_demo.txt"

# 1. Shrinking a File
print("--- Shrinking File via truncate() ---")
# Write 20 bytes initially
with open(truncate_file, "w+", encoding="utf-8") as f:
    f.write("12345678901234567890")
    f.flush()  # Force write
    
    # Truncate to 10 bytes
    f.truncate(10)
    
    # Read content
    f.seek(0)
    print(f"Content after truncate(10): {repr(f.read())}")
    # Expected: '1234567890' (Discarded last 10 bytes)



In [ ]:
# 2. Expanding a File (Null-byte padding)
print("\n--- Expanding File via truncate() ---")
with open(truncate_file, "r+b") as f:  # Open in binary mode to inspect null bytes
    # File is currently 10 bytes. Expand to 15 bytes.
    f.truncate(15)
    f.seek(0)
    content = f.read()
    print(f"Content bytes:     {content}")
    print(f"Integers list:     {list(content)}")
    # Expected integers end with: [..., 48, 0, 0, 0, 0, 0] (48 is ASCII '0')



In [ ]:
# 3. Truncating at current pointer position
print("\n--- Truncating at Current Pointer ---")
with open(truncate_file, "r+b") as f:
    f.seek(5)
    # Truncates at index 5. Elements after index 5 are discarded.
    f.truncate()
    f.seek(0)
    print(f"Remaining bytes: {f.read()}")  # Expected: b'12345'



In [ ]:
# 4. Clean up
if os.path.exists(truncate_file):
    os.remove(truncate_file)
